# Asking the graph

Two engines read the same graph, and they are good at opposite things.

**AQLizer** is Arango's natural-language-to-AQL service. It writes a query, runs it,
and hands back both the rows and the query -- so an answer can be checked rather
than trusted. It is the one to use when the question has a shape: count, sum, rank,
traverse, or *find the ones that are missing something*.

**GraphRAG** is the retriever. It searches the entity descriptions and the source
text, follows the relations it lands on, and writes an answer from what it read.
It is the one to use when the question has no shape -- when it is phrased in words
the model does not use, or when the answer is spread across a dozen files.

Neither is a fallback for the other. The last section asks one question both ways
to show where the line is.

In [1]:
import logging
from sysml import nl

logging.disable(logging.INFO)  # both services narrate every step

---

# Part 1 -- AQLizer

Nothing below is a hand-written query. `sysml/aql_examples.md` teaches the model how
SysML concepts are laid out here -- an `attributes` map, an `owns`/`typedby` tree, a
`stated` flag on the edges -- and the AQL in every answer is what it wrote from that.

## 1. A mass budget

This is the hardest shape in the set: walk up to six hops down the containment tree
through two different edge types, pull two attributes off each element it lands on,
add them, sort by the sum, and cite where each number is declared.

In [2]:
nl.instance().ask(
    "For each Saturn V stage, give its dry mass, its propellant mass and the sum of "
    "the two, sorted by the total, with the file and line each is declared on."
).show(row_limit=7)

Q  For each Saturn V stage, give its dry mass, its propellant mass and the sum of the two, sorted by the total, with the file and line each is declared on.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.entity_name == "SATURNV"
     LET parts = (
       FOR child, edge IN 1..6 OUTBOUND e sysml_Relations
         FILTER edge.relationship_type IN ["owns", "typedby"]
         FILTER child.attributes.dryMass.value != null AND child.attributes.propellantMass.value != null
         LET dryMass = child.attributes.dryMass.value
         LET propellantMass = child.attributes.propellantMass.value
         LET totalMass = dryMass + propellantMass
         RETURN {
           name: child.entity_name,
           dryMass: dryMass,
           propellantMass: propellantMass,
           totalMass: totalMass,
           unit: child.attributes.dryMass.unit,
           at: CONCAT(child.source_file, ":", child.source_line)
         }
     )
     SORT parts[*].totalMas

Every figure is a number a file states, and the `file:line` beside it is where. That
is the half of the graph the lexer wrote; an LLM reading the same text reports the
masses as prose and cannot be summed.

## 2. What is *not* there

Coverage questions are the ones a requirements engineer actually asks, and they are
an anti-join: requirements with no incoming `satisfies` edge. Retrieval cannot answer
this at all -- there is no passage describing the absence of a relation.

In [3]:
nl.instance().ask(
    "Which ten Apollo requirements have the most elements satisfying them, and how "
    "many Apollo requirements have none at all?"
).show(row_limit=3)

Q  Which ten Apollo requirements have the most elements satisfying them, and how many Apollo requirements have none at all?

AQL
   WITH sysml_Entities, sysml_Relations
   LET apolloRequirements = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
       FILTER e.source_file != null
       RETURN e
   )
   LET topSatisfiedRequirements = (
     FOR e IN apolloRequirements
       LET satisfiers = (
         FOR r IN sysml_Relations
           FILTER r._to == e._id AND r.relationship_type == 'satisfies'
           AND r.stated == true
           RETURN 1
       )
       LET count = LENGTH(satisfiers)
       SORT count DESC
       LIMIT 10
       RETURN {requirement: e.entity_name, satisfiers: count}
   )
   LET unsatisfiedRequirementsCount = LENGTH(
     FOR e IN apolloRequirements
       LET satisfiers = (
         FOR r IN sysml_Relations
           FILTER r._to == e._id AND r.relationship_type == 'satisfies'
           AND r

## 3. The graph can be asked how it was built

Every relation carries `stated`: true if the lexer read it out of the syntax, absent
if the LLM inferred it. So "how much of this graph is read and how much is guessed"
is itself a query -- per model, in one pass.

In [4]:
nl.instance().ask(
    "Break the relations down by model and by whether they were read from the "
    "syntax or inferred by the LLM."
).show(row_limit=6)

Q  Break the relations down by model and by whether they were read from the syntax or inferred by the LLM.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR r IN sysml_Relations
     FILTER r.type == "RELATED_TO"
     LET from = DOCUMENT(r._from)
     FILTER from != null
     FOR model IN from.models
       COLLECT m = model, read = r.stated == true WITH COUNT INTO n
       RETURN {model: m, source: read ? "read from the syntax" : "inferred by the LLM", relations: n}

rows (6, first 6)
   {"model": "apollo-11-sysml-v2", "source": "inferred by the LLM", "relations": 564}
   {"model": "apollo-11-sysml-v2", "source": "read from the syntax", "relations": 2938}
   {"model": "Drone_BaseArchitecture", "source": "inferred by the LLM", "relations": 2}
   {"model": "Drone_BaseArchitecture", "source": "read from the syntax", "relations": 8}
   {"model": "DroneModelLogical", "source": "inferred by the LLM", "relations": 5}
   {"model": "DroneModelLogical", "source": "read from the syntax", "rela

The read majority is the point of the second pass. What the LLM contributes is the
571 relations that are in the *prose* -- `refines`, `dependson`, `performs` -- which
no lexer can find.

## 4. One identifier, two elements

`HLR-R001` is written twice in this corpus and means something different each time.
`requirement def <'HLR-R001'> CrewReturnSafetyRequirement` declares the requirement
and carries the identifier as a *short name*; `requirement 'HLR-R001' :
CrewReturnSafetyRequirement` uses it in a specification, and there the identifier is
the element's *name*. Two rows, and the edges divide between them along a line an
engineer would care about: the definition takes the `refines` from the requirements
below it, and the usage is what every `satisfy 'HLR-R001' by ...` statement points
at.

Matching one field, or taking `FIRST` of the match, returns half the picture and
looks complete. Matching both is the whole element.

In [5]:
nl.instance().ask(
    "Trace HLR-R001: what satisfies it, what refines it, and what it is related "
    "to in either direction."
).show(row_limit=8)

Q  Trace HLR-R001: what satisfies it, what refines it, and what it is related to in either direction.

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER e.short_name == "HLR-R001" OR e.entity_name == "HLR-R001"
     FOR v, r IN 1..1 ANY e sysml_Relations
       FILTER r.type == 'RELATED_TO'
       RETURN {of: e.entity_name, relation: r.relationship_type, other: v.entity_name,
               direction: r._from == e._id ? 'outgoing' : 'incoming',
               stated: r.stated == true, description: r.description}

rows (28, first 8)
   {"of": "HLR-R001", "relation": "satisfies", "other": "RECOVERYQUARANTINEOPERATIONS_RETRIEVECREWANDCM", "direction": "incoming", "stated": true, "description": "apollo11Mission.apollo11Phases.recoveryQuarantine.recoveryQuarantineOperations.retrieveCrewAndCM satisfies hlr-R001"}
   {"of": "HLR-R001", "relation": "owns", "other": "APOLLO11MISSIONSPECIFICATION", "direction": "incoming", "stated": true, "description": "MissionS

## 5. Joining a layer that is not in any file

The `SIMILAR_TO` edges are computed, not declared -- autograph's `SimilarityFinder`
matching entities of the same kind across model boundaries. They are queryable like
anything else, so "what does the drone have in common with Apollo" is a join.

In [6]:
nl.instance().ask(
    "Which requirements does the drone model state that the Apollo model has an "
    "analogous requirement for, and how close are they?"
).show(row_limit=6)

Q  Which requirements does the drone model state that the Apollo model has an analogous requirement for, and how close are they?

AQL
   WITH sysml_Entities, sysml_Relations
   FOR e IN sysml_Entities
     FILTER 'DroneModelLogical' IN e.models
     FILTER e.entity_type == 'requirement'
     FILTER e.source_file != null
     FOR v, r IN 1..1 ANY e sysml_Relations
       FILTER r.type == 'SIMILAR_TO'
       FILTER 'apollo-11-sysml-v2' IN v.models
       FILTER v.entity_type == 'requirement'
       RETURN {drone_requirement: e.entity_name, apollo_requirement: v.entity_name, cosine: r.cosine}

rows (12, first 6)
   {"drone_requirement": "DRONEENGINESTANDARDSTAKEHOLDERREQUIREMENTS_RELIABILITY", "apollo_requirement": "MISSIONSYSTEMRELIABILITY", "cosine": 0.6049595866485458}
   {"drone_requirement": "DRONEENGINESTANDARDSTAKEHOLDERREQUIREMENTS_CONTROL", "apollo_requirement": "MISSIONCONTROL_OPERATIONALCOORDINATION", "cosine": 0.5583265137238257}
   {"drone_requirement": "DRONEENGINESTANDARDST

### Where AQLizer stops

It needs the question to land on a field. Ask it something whose answer is spread
through the `doc` comments of a dozen requirements in four files and there is no
column to filter on -- which is the next section.

---

# Part 2 -- GraphRAG

Three scopes, all upstream, all reading this graph.

  `local`    hybrid vector + BM25 over the entities, fused, then expanded over the
             relations it lands on
  `unified`  the source chunks and the entity graph searched in parallel
  `global`   the community reports, map-reduced

## 6. `local` -- a question in words the model never uses

No SysML file contains "alive", "breathing" or "keeps". The elements are called
`PLSS`, `PSA`, `EnvironmentalControlSystem`. Vector search does not care.

In [7]:
(await nl.retriever().ask_async(
    "What keeps the astronauts alive and breathing, and what limits does it "
    "have to hold?"
)).show(row_limit=4)

Q  What keeps the astronauts alive and breathing, and what limits does it have to hold?

retrieved  29 documents, 69 edges, 80,853 chars of context

cited (9, first 4)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Purpose/StakeholderPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/StakeholderNeedsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}

A  ## Systems Keeping Astronauts Alive and Breathing

### Cabin Environmental Control Requirement
The requirement, identified as `CLR-R051`, mandates that the spacecraft's environmental control system maintains the cabin temperature between 20°C and 25°C and oxygen partial pressure between 2.5 psi and 4.0 psi throughout the entire 8-day mission for a crew of three [CITE:6]. By controlling the cabin temperature and oxygen levels, the environ

## 7. `unified` -- a figure that never became an entity

Some numbers live only in a `doc` comment, so they are in the source text and in no
`attributes` map. `unified` searches the chunks and the graph together, which is what
reaches them.

The second call is the important one: `evidence(find=...)` prints the retrieved text
around the figure, so the answer can be checked against what was actually read rather
than taken on trust.

In [8]:
answer = await nl.retriever().ask_async(
    "How much drinking water must the environmental control system supply "
    "per crew member per day?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=420, find="water")

Q  How much drinking water must the environmental control system supply per crew member per day?

retrieved  12 documents, 79 edges, 29,514 chars of context

cited (7, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Requirements/FunctionalRequirementsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Answer

The environmental control system for the Apollo 11 Mission, as specified in the SysML model, must supply a minimum of 2 kilograms of potable water per crew member per day [CITE:1].

evidence  (420 chars at char 2,689, of 29,514 retrieved)
   		}
   	}
   	requirement def <'CLR-R064'> ECSWaterSupplyRate {
   		doc /* The Apollo 11 Mission's crew's potable water system shall provide a minimum of 2 kilograms of potable water per crew member per day. */
   		@Rationale {
   			text = "Adequate potabl

The number in the answer is in the `doc` comment printed underneath it, and the
citation resolves to the file it came from. That is the difference between a
retrieval and a recollection.

In [9]:
answer = await nl.retriever().ask_async(
    "What is the minimum delta-v the lunar module ascent stage has to provide, "
    "and why that figure?",
    scope="unified")
answer.show(row_limit=3)
answer.evidence(chars=320, find="delta")

Q  What is the minimum delta-v the lunar module ascent stage has to provide, and why that figure?

retrieved  7 documents, 42 edges, 16,267 chars of context

cited (6, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}

A  ## Minimum Delta-V Requirement for Lunar Module Ascent Stage

### Requirement Details

The minimum delta-V that the Lunar Module ascent stage must provide is 1,850 m/s. This requirement is specified under the designation `CLR-R115` and is fundamental for the Lunar Module (LM) to achieve lunar orbit after ascent from the Moon's surface. The rationale provided is that it specifies the necessary propulsive performance to escape the Moon's gravity, ensuring that the LM can successfully enter lunar orbit[CITE:1][CITE:

## 8. `global` -- a question no single row answers

`global` never looks at an entity. It reads the 280 community reports the extraction
step wrote, scores them against the question, and summarises the ones that survive --
so it answers about the corpus as a whole.

In [10]:
(await nl.retriever().ask_async(
    "What concerns are these models organised around, and what does each part "
    "of the corpus contribute?",
    scope="global")).show()

Q  What concerns are these models organised around, and what does each part of the corpus contribute?

retrieved  74 community reports -> 51 points

A  # Overview of Concerns and Contributions in the Models

This response synthesizes information from a dataset of SysML v2 models related to the Apollo 11 mission, focusing on the organizational concerns and contributions of each part.

## Core Mission Elements and Specifications

1. **Apollo11Mission Core Elements** (Reported by Analyst 0):
    - **Concerns**: Mission specifications, objectives, lifecycle evaluations, and cost attributes.
    - **Contribution**: Define the mission requirements crucial for understanding the entire mission framework.

2. **ApolloCommandServiceModule and Its Components** (Reported by Analyst 0):
   - **Concerns**: Energy coordination, intermodule connectivity, subsystem interactions, and historical specialization.
   - **Contribution**: Highlight inter-module connectivity and energy management within the Ap

---

# The line between them

One question, both engines.

In [11]:
QUESTION = "How many Apollo requirements does nothing satisfy?"

nl.instance().ask(QUESTION).show(row_limit=2)

Q  How many Apollo requirements does nothing satisfy?

AQL
   WITH sysml_Entities, sysml_Relations
   LET unsatisfied = (
     FOR e IN sysml_Entities
       FILTER e.entity_type == 'requirement' AND 'apollo-11-sysml-v2' IN e.models
       FILTER e.source_file != null
       FILTER LENGTH(FOR r IN sysml_Relations
                FILTER r._to == e._id AND r.relationship_type == 'satisfies'
                AND r.stated == true
                RETURN 1) == 0
       RETURN e.entity_name)
   RETURN {total: LENGTH(unsatisfied), examples: SLICE(unsatisfied, 0, 10)}

rows (1, first 1)
   {"total": 330, "examples": ["MISSION_GOALS", "CAPABILITYTOGOALDERIVATION_GOALS", "NEEDS", "APOLLO11MISSION_DEPLOY", "NASA_MISSIONSUCCESS", "NASA_ASTRONAUTSAFETY", "NASA_PROGRAMMATICFUNDING", "NASA_INTERNATIONALPRESTIGE", "NASA_SCIENTIFICRETURN", "NASA_PLANETARYPROTECTION"]}

A  There are 330 Apollo requirements that are not satisfied by anything. Some examples of these unsatisfied requirements include 'MISSION

In [12]:
(await nl.retriever().ask_async(QUESTION)).show(row_limit=3)

Q  How many Apollo requirements does nothing satisfy?

retrieved  31 documents, 113 edges, 99,570 chars of context

cited (6, first 3)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Purpose/MissionSpecificationPackage.sysml"}

A  The model does not say how many Apollo requirements nothing satisfies. The context focuses on the requirements and their satisfactions by different components of the Apollo 11 mission system, but it does not specify a requirement or a number of requirements that are explicitly unsatisfied by any component.



AQLizer counts them. GraphRAG cannot: it retrieves requirements that *look* relevant
and describes them, because there is no passage anywhere that states how many
requirements lack a relation. Reverse the question -- "what keeps the astronauts
alive" -- and AQLizer has nothing to filter on while GraphRAG answers from four files.

So the rule is about the question, not the engine:

| the question is about | use |
|---|---|
| a number, a count, a ranking, a rollup | AQLizer |
| something absent -- unsatisfied, unowned, uncovered | AQLizer |
| provenance, or the shape of the graph itself | AQLizer |
| a concept the model spells differently | GraphRAG `local` |
| a figure written in prose rather than declared | GraphRAG `unified` |
| the corpus as a whole | GraphRAG `global` |

Both are pointed at a graph the importer's own writer produced, and neither has a
hand-written query behind it. When an answer is wrong, the fix goes in
`sysml/aql_examples.md` -- two of the queries above are only correct because a
previous wrong answer was turned into a worked example there.